In [1]:
import unitaria as ut
import numpy as np

/homes/wiss/deimmatt/projects/unitaria/.venv/lib/python3.12/site-packages/excitationsolve/excitation_solve_scipy.py:44: SyntaxWarning: invalid escape sequence '\p'
  expects the parameters to be 2\pi periodic. For example, in Qiskit


## Example: Computing the energy of a finite element function

Block encodings can in principle be used for the solution of vector-valued problems, but are only more efficient than classical computing techniques when the dimension is very large. In this thesis, we are especially interested in the solution of PDEs, which is classically realized by the solution of such large linear systems. The dimension of this linear system depends on the accuracy one wants to obtain for the solution, making PDEs ideal candidates for application of block encodings.

A prototypical example for this is the $1$-dimensional Poisson equation, which models diffusion or heat transfer processes. For a unit interval domain $D = [0, 1]$ it is given by
\begin{align*}
    -\Delta u &= f && \text{in } D \\
    u &= 0 && \text{on } \partial D.
\end{align*}
Here, we incorporate homogeneous Dirichlet boundary conditions, which restrict the solution $u \colon D \to \mathbb{R}$ at $x = 0$ and $x = 1$. The function $f \colon D \to \mathbb{R}$ corresponds to a source that, e.g., continuously adds heat to the system.

We recall that the corresponding "stiffness matrix" is given by
$$
S =
N\begin{bmatrix}
    2 &  -1&& \\
    -1 & \ddots &\ddots& \\
    & \ddots & \ddots&-1 \\
    &&-1&2
\end{bmatrix}
= N(2\mathrm{Id}_{N-1} - L - L^\top), \quad\text{where}\quad L \coloneqq
\begin{bmatrix}
    0 &  && \\
    1 & \ddots && \\
    & \ddots & \ddots& \\
    &&1&0
\end{bmatrix}.
$$
In `unitaria`, we can write the following code for this, where we set the discretization parameter to $N = 2^n$ with $n = 4$.

In [2]:
n = 4
N = 2**n
Inc = ut.Increment(bits=n)
L = Inc[:N - 1, :N - 1]
S = N * (2 * ut.Identity(dim=N - 1) - L - L.adjoint())

The meaning of the stiffness matrix is essentially that the solution $u$ of the Poisson equation is approximated by the solution $c$ of a linear system of the form
$$
Sc = r
$$
with some $r \in \mathbb{R}^N$ that encodes $f$. As an example we use
$$
r = [1 \; 0 \; 1 \; \dots \; 0 \; 1]^\top.
$$
An efficient encoding of this vector is given via the decomposition
$$
r = \tilde r|_{N-1}, \quad\text{where}\quad \tilde{r} \coloneqq [1\; \dots\; 1]^\top \otimes [1 \; 0]^\top  \in \mathbb{R}^{N}.
$$
The tensor product operation corresponds to the `&` operator in `unitaria`. The restriction to the first $N-1$ components is implemented using the index syntax `[:N - 1]`, which one would also use for `NumPy` arrays.

In [3]:
r = (
    ut.Ones(2 ** (n - 1))
    & ut.ConstantVector(np.array([1, 0]))
)[:N - 1]

We are, however, not yet ready to solve linear systems, which will be topic of the following chapter. Instead, let us compute an inner product of the form
$$
q \coloneqq r^\top S r
$$
which one would call the "energy" of the function represented by $r \in \mathbb{R}^N$.
We can compute the inner product as a block encoding.

In [4]:
q = r.adjoint() @ S @ r

Finally, we can use this block encoding to extract the encoded value ...

In [5]:
print(q.compute_norm())

256.0


... or simulate its estimation.

In [6]:
estimator = ut.Simulator(
    scheme="phase-estimation",
    default_precision=q.normalization * 0.05,
    default_failure_probability=0.01,
    count_gates=True,
    seed=1234
)
print(estimator.estimate_norm(q))
print(estimator.gate_count)

268.1809704508782
{'s': 6960096, 'h': 10967616, 'x': 1037520, 't': 9459648, 'cx': 918720, 'z': 3168, 't-depth': 5673888}
